In [ ]:
#imports
import pandas as pd
import numpy as np

In [ ]:
#making decimal points only two
pd.options.display.float_format = '{:20,.2f}'.format

In [ ]:
# TODO: what stories were made from these?
# TODO: fish comment about which days there weren't any arrests, pulling out holidays
# TODO: get Flourish login
# TODO: get style guide
# TODO: longer term data is available, do we want to make use of it?
# TODO: did we use the processed or original data?
# TODO: pay attention to duplicate likely row
# TODO: I think we should start open-sourcing our methodology/scripts

## Source

[Deportation Data Project](https://deportationdata.org/data/ice.html)

## Research Questions

* How many people were deported to a third country?
* How many people had a criminal record?


# Arrests
This is the main table we are working with to start. From the description: "Records every time ICE arrests someone, whether or not that arrest results in a decision to detain the person; or issues a Notice to Appear (NTA), the document that starts a deportation case. Note that other agencies also issue NTAs, and those would not appear as arrests in this table. We treat "apprehensions," "arrests," and "administrative arrests" as synonyms."

In [ ]:
# TODO: where do we get this?
arrests = pd.read_csv('arrests.csv')
arrests.info()

In [ ]:
#This gets arrests with Denver area of responsibility and/or the state of Colorado/Wyoming
co_arrests = arrests[
    (arrests['Apprehension AOR'] == 'Denver Area of Responsibility') |
    (arrests['Apprehension State'] == 'Colorado') | (arrests['Apprehension State'] == 'Wyoming')
]
co_arrests.info()

In [ ]:
co_arrests.to_csv('colo_arrests.csv')

# Detentions
This is the detentions section. The description: "Records each detention from book-in to book-out at a given detention center for a given individual; most individuals have more than one row in the table because they are transferred between detention centers. Individuals may also be detained more than once and therefore have more than one "stay" in detention. We explain further in the relevant field definitions."
This spreadsheet has two tables that must be combined.

In [ ]:
#import two detention tables
detentions_early = pd.read_csv('detention_early.csv')
detentions_recent = pd.read_csv('detention_later.csv')

In [ ]:
#create a list of prior two tables, then concantenate them to one; check the record totals
pdList = [detentions_early, detentions_recent]  
detentions = pd.concat(pdList)
detentions.info()

In [ ]:
#this gets a list of the detention facilities
det_facility = detentions.groupby(['Detention Facility']).size().reset_index(name='counts')
det_facility.to_csv('det_facility.csv')

In [ ]:
#these are the colorado, wyoming facilities thus far
# TODO: where do we get this?
my_facilities = [
    'ALAMOSA HOLDROOM',
    'ARAPAHOE COUNTY JAIL',
    'AURORA CITY JAIL',
    'CASPER HOLDROOM',
    'CHEYENNE HOLDROOM',
    'COLO DEPT OF CORRECTIONS',
    'COLO SPRINGS DEN HSI HOLD',
    'CRAIG HOLDROOM',
    'DENVER CONTRACT DETENTION FACILITY',
    'DENVER COUNTY JAIL',
    'DENVER HEALTH MEDICAL CENTER',
    'DENVER HOLD ROOM',
    'DOUGLAS COUNTY DETENTION CENTER',
    'DURANGO HOLDROOM',
    'GLENWOOD SPRINGS HOLDROOM',
    'GRAND JUNCTION HOLDROOM',
    'JACKSON COUNTY SHERIFF',
    'JEFFERSON COUNTY JAIL',
    'LARAMIE COUNTY JAIL',
    'MESA COUNTY JAIL',
    'MOFFAT COUNTY JAIL',
    'NATRONA COUNTY JAIL',
    'OTERO COUNTY DETENTION',
    'PUEBLO COUNTY JAIL',
    'PUEBLO HOLDROOM',
    'SUMMIT COUNTY JAIL',
    'SWEETWATER COUNTY JAIL',
    'TELLER COUNTY JAIL', 
    'UCHEALTH UNV HOSP OF CO'
    'UINTA COUNTY JAIL'
    
]

In [ ]:
#this uses the list of colo, wyo to extract the info from the larger detention facility list
co_wyo_detain = detentions[detentions['Detention Facility'].isin(my_facilities)]
co_wyo_detain.info()

In [ ]:
co_wyo_detain.to_csv('co_wyo_detain.csv')

In [ ]:
#get a table with the unique identifer and the charge
charge_lookup = detentions[['Unique Identifier', 'MSC Charge']].drop_duplicates(subset='Unique Identifier')
charge_lookup.info()

In [ ]:
#adds most serious conviction information, if present, to the arrests table
arrests_with_charge = co_arrests.merge(charge_lookup, on='Unique Identifier', how='left')
arrests_with_charge.info()

In [ ]:
arrests_with_charge.to_csv('arrests_charges.csv')

## Removals
From the description: "Records every deportation that ICE conducts, with a row for each individual deportation. An individual only has more than one row if that individual was deported more than once. Note that expulsions may occur directly at the border, by CBP, without involving ICE."

In [ ]:
removals = pd.read_csv('removal.csv')
removals.info()

In [ ]:
co__wyo_remove = removals[
    (removals['Docket AOR'] == 'Denver Area of Responsibility') |
    (removals['Apprehension State'] == 'Colorado') |
    (removals['Apprehension State'] == 'Wyoming')
]

In [ ]:
co__wyo_remove.info()

In [ ]:
co__wyo_remove.to_csv('co_wyo_remove.csv')

# Detainers
From the description: "Records all requests to state, county, and municipal jails and prisons either for a person to be held on a detainer or for a notification of release date and time. A detainer is a request to a local jail to hold someone for 48 hours beyond when they otherwise would be released so that ICE can make an arrest in the jail while the individual remains detained."

In [ ]:
detainer = pd.read_csv('detainers.csv')
detainer.info()

In [ ]:
felon_lookup = detainer[['Unique Identifier', 'MSC Conviction Date', 'Detention Facility','Facility State', 'Prior Felony Yes No', 'Violent Misdemeanor Yes No','Aggravated Felony Yes No']].drop_duplicates(subset='Unique Identifier')

In [ ]:
felon_lookup.info()

In [ ]:
#this adds more information about conviction date, facility and more; mostly used the conviction date
arrests_supplement = arrests_with_charge.merge(felon_lookup, on='Unique Identifier', how='left')
arrests_supplement.info()


In [ ]:
arrests_supplement.to_csv('arrests_plus.csv')

In [ ]:
detainer.groupby(['Facility AOR']).size().reset_index(name='counts')

In [ ]:
detainer.groupby(['Facility State']).size().reset_index(name='counts')

In [ ]:
co_wyo_detainer = detainer[
    (detainer['Facility AOR'] == 'Denver Area of Responsibility') |
    (detainer['Facility State'] == 'Colorado') |
    (detainer['Facility State'] == 'Wyoming')
]
co_wyo_detainer.info()

In [ ]:
co_wyo_detainer.to_csv('co_wyo_detainer.csv')

# Encounters
From the description: "Records every time ICE Enforcement and Removal Operations encounters a person, i.e. considers whether to take enforcement action against a person. This need not mean a physical encounter. Most notably, every time ICE processes a match between FBI book-in information (i.e. to a jail or prison) and ICE database information, that match is logged as an ICE encounter. Generally, if an individual appears in the detainers or arrests table, that individual should appear in this table. An individual might appear in the removals or detentions tables without appearing in the encounters data if Customs and Border Protection initially encounters the person. This is both the largest and the sparsest of the tables, and in many cases, encounters lack a unique ID because the individual lacked an A number (A numbers are generally only given to people with immigrant visas or when they are processed for deportation proceedings)."

In [ ]:
encounters_early = pd.read_csv('encounters_early.csv')
encounters_later = pd.read_csv('encounters_later.csv')

In [ ]:
pdList = [encounters_early, encounters_later]  
encounters = pd.concat(pdList)
encounters.info()

In [ ]:
encounters.groupby(['Responsible AOR']).size().reset_index(name='counts')

In [ ]:
co_encounter = encounters[encounters['Responsible AOR'] == 'Denver Area of Responsibility']
co_encounter.info()

In [ ]:
co_encounter.to_csv('co_encounter.csv')

In [ ]:
#have not looked at this closely
encounters.groupby(['Responsible Site']).size().reset_index(name='counts')

## What's next
Use the arrests_plus.csv file to clean the data in excel.

First filter for blanks in the Apprehension State field. See if Wyoming or Colorado show up in the Facility State, and then if Detention Facility or Apprehension Landmark Site appear to be in Wyoming or Colorado, and update the Apprehension State accordingly.

Then take a look at the other states listed in the Apprehension State field and remove those that don't belong. (More on this later.)

Then search for unique identifiers using pivot table counts and examine those that are more than one. If they occurred in different years, keep both. If they occurred within a day or two get rid of the older one (often there's removal on one of them, keep that one).

If you plan to compare 2025 to 2025 numbers, take out only the data for the dates available in 2025 (Jan. 20 to the most recent month and date). Don't forget to take out the records from Jan.1-19, 2025.

Then it's time for analysis. (will update with a new notebook at some point.)

In [ ]:
# TODO: transfer the excel cleaning steps to python